# Generate synthetic times

In [1]:
import numpy as np
from sim_tools.distributions import Lognormal
import pandas as pd

In [2]:
CSV_PATH = "../data/times.csv"

np.random.seed(42)
n = 100_000

# Category and outcome labels to sample from
cats = ["Category 1", "Category 2", "Category 3", "Category 4"]
outcomes = ["See & Treat", "See & Convey ED", "See & Convey non ED"]

# Build the base dataframe
# np.random.choice with no p argument assigns equal probability to each option
df = pd.DataFrame(
    {
        "ResponseCategoryGroupLevel2": np.random.choice(cats, n),
        "C0660_CallOutcomeDetail": np.random.choice(outcomes, n),
    }
)

# Target mean and sd for each duration (no upper boundary)
durations = {
    "C0001_Mobilisation_Duration": {"mean": 20, "sd": 10},
    "C0004_MobilisationToScene_Duration": {"mean": 200, "sd": 80},
    "C0007_OnScene_Duration": {"mean": 800, "sd": 300},
    "C0172_SceneToDestination_Duration": {"mean": 300, "sd": 150},
    "C0012_Handover_Duration": {"mean": 400, "sd": 200},
    "C0171_WrapUp_Duration": {"mean": 150, "sd": 80},
}

# Simple flat shift per category to make each look a little different
cat_shift = {
    "Category 1": 0,
    "Category 2": 10,
    "Category 3": 20,
    "Category 4": 30,
}

for col, params in durations.items():
    # Sample from lognormal distribution
    base = Lognormal(mean=params["mean"], stdev=params["sd"]).sample(size=n)
    # Shift by category
    shift = df["ResponseCategoryGroupLevel2"].map(cat_shift)
    df[col] = (base + shift).round(1)

In [3]:
df.head()

,ResponseCategoryGroupLevel2,C0660_CallOutcomeDetail,C0001_Mobilisation_Duration,C0004_MobilisationToScene_Duration,C0007_OnScene_Duration,C0172_SceneToDestination_Duration,C0012_Handover_Duration,C0171_WrapUp_Duration
0,Category 3,See & Treat,44.4,200.4,431.7,470.9,161.1,254.8
1,Category 4,See & Convey ED,55.0,124.7,1930.1,419.9,329.8,196.8
2,Category 1,See & Convey ED,6.5,95.7,473.9,352.0,607.8,126.2
3,Category 3,See & Convey ED,37.9,358.6,1608.0,466.6,806.9,243.1
4,Category 3,See & Treat,37.8,289.3,832.2,344.0,184.1,73.1


In [4]:
df.to_csv(CSV_PATH, index=False)